# CloudGraph API Guide

This notebook demonstrates how to interact with the CloudGraph FastAPI backend, specifically focusing on the photo upload flow with EXIF processing and S3 storage.

## 1. Setup Configuration
Configure your local variables below. You will need a valid Cognito token to authenticate.

In [ ]:
import requests
import json

# Set your configurations here
API_BASE_URL = "http://localhost:8000/api"

# Place a valid Cognito ID token or Access token here
AUTH_TOKEN = "YOUR_COGNITO_TOKEN_HERE"

# Note: The file should be a valid JPEG, PNG, or HEIC image
# Make sure to place a sample photo in the same directory as this notebook.
IMAGE_PATH = "sample_photo.jpg"

headers = {
    "Authorization": f"Bearer {AUTH_TOKEN}"
}

## 2. Uploading a Photo
We use standard `multipart/form-data` to upload the image file. The backend will:
1. Validate your Cognito JWT token.
2. Extract EXIF metadata (Date Taken, GPS Coordinates).
3. Upload the raw image bytes to AWS S3.
4. Return the new `file_key`, a 1-hour `presigned_url`, and the extracted `metadata`.

In [ ]:
upload_endpoint = f"{API_BASE_URL}/upload"

try:
    with open(IMAGE_PATH, "rb") as f:
        files = {"file": (IMAGE_PATH, f, "image/jpeg")}
        
        print(f"Uploading {IMAGE_PATH} to {upload_endpoint}...")
        response = requests.post(upload_endpoint, headers=headers, files=files)
        
        # The backend response will be JSON
        response_data = response.json()
        
        if response.status_code == 200:
            print("\n\u2705 Upload Successful!")
            print(json.dumps(response_data, indent=2))
            
            # Retrieve presigned URL for the uploaded S3 object
            presigned_url = response_data.get("presigned_url")
            file_key = response_data.get("file_key")
        else:
            print(f"\n\u274c Upload Failed with status code {response.status_code}")
            print(response_data)
            presigned_url = None
            
except FileNotFoundError:
    print(f"\u274c Could not find '{IMAGE_PATH}'. Please ensure you have a sample photo in the directory.")
    presigned_url = None
except Exception as e:
    print(f"\u274c Request failed: {e}")
    presigned_url = None

## 3. Retrieving the Image
Because the storage is protected and private, you can download the file temporarily from AWS using the `presigned_url` generated by the backend.

In [ ]:
import io
from PIL import Image
from IPython.display import display

if 'presigned_url' in locals() and presigned_url:
    print("Downloading image from S3 using presigned URL...")
    img_response = requests.get(presigned_url)
    
    if img_response.status_code == 200:
        print("\u2705 Success! Displaying image:\n")
        img = Image.open(io.BytesIO(img_response.content))
        
        # Resize to display nicely in the notebook without zooming
        img.thumbnail((400, 400))
        display(img)
    else:
        print(f"\u274c Failed to fetch image from S3. HTTP {img_response.status_code}")
else:
    print("\u26a0\ufe0f No presigned URL available. Please run the upload step successfully first.")